In [ ]:
# Start with AMI
from shapely.geometry import shape, Point
import geopandas as gpd

# Re-read the original GeoJSON file
geojson_path = "/mnt/data/Main Street Dryden.geojson"
with open(geojson_path) as f:
    geojson_data = json.load(f)

# Extract polygon geometries
geojson_geoms = [shape(feature["geometry"]) for feature in geojson_data["features"]]

# Extract AMI coordinate points with servicepointid
ami_coords = [
    (6000990168, 42.49119807, -76.29978434),
    (6000981995, 42.49452236, -76.30406455),
    (6000998944, 42.49173932, -76.27344414),
    (6000990452, 42.49716834, -76.27910346),
    (6000981895, 42.4948901, -76.30532225),
    (6000999930, 42.4905579, -76.2986098),
    (6000983779, 42.48898196, -76.31015417),
    (6000999681, 42.49034847, -76.31261879),
    (6000984893, 42.48960364, -76.29499341)
]

# Find nearest footprint for each AMI coordinate
assigned_features = []
used_indices = set()

for sid, lat, lon in ami_coords:
    pt = Point(lon, lat)
    distances = [pt.distance(poly) if i not in used_indices else float("inf") for i, poly in enumerate(geojson_geoms)]
    min_idx = distances.index(min(distances))
    used_indices.add(min_idx)

    feature = {
        "type": "Feature",
        "geometry": mapping(geojson_geoms[min_idx]),
        "properties": {"servicepointid": sid}
    }
    assigned_features.append(feature)

# Write the updated GeoJSON
final_geojson = {
    "type": "FeatureCollection",
    "features": assigned_features
}

final_path = "/mnt/data/Assigned_AMI_to_Footprints.geojson"
with open(final_path, "w") as f:
    json.dump(final_geojson, f)

final_path
